# Phase 1 - SQL Analytics Layer

## Business Intelligence Foundation

This notebook is the Phase 1 SQL deliverable for the hospital analytics capstone. It builds a reliable SQLite analytics layer from the raw hospital CSV files and presents the required operational, financial, and data-quality results.

Business goal: create a reliable, queryable hospital data layer that leadership can trust for operational and financial decision-making.

Technical objectives:

- Transform raw CSV files into a structured relational database.
- Define `patients`, `visits`, and `billing` tables with appropriate data types.
- Enforce primary key and foreign key relationships.
- Create reusable business views.
- Add indexes for frequent joins, filters, and aggregations.
- Answer the required SQL analysis questions.


## Source Datasets

The notebook expects these files in the same folder:

- `patients.csv`
- `visits.csv`
- `billing.csv`

The generated database is `hospital_analytics.db`.


In [ ]:
from pathlib import Path
import sqlite3
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'patients.csv').exists() and (ROOT.parent / 'patients.csv').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'hospital_analytics.db'

for file_name in ['patients.csv', 'visits.csv', 'billing.csv']:
    with open(ROOT / file_name, newline='') as handle:
        row_count = sum(1 for _ in handle) - 1
    print(f'{file_name}: {row_count:,} rows')


## Build the SQL Database

This cell rebuilds the database from the CSV files. The build script creates raw staging tables, final constrained relational tables, indexes, and reusable views.


In [ ]:
result = subprocess.run(
    ['python3', 'scripts/build_sqlite_db.py'],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)


## Schema, Keys, and Relationships

The schema below defines the hospital relational model.

Primary keys:

- `patients.patient_id`
- `visits.visit_id`
- `billing.bill_id`

Foreign keys:

- `visits.patient_id` references `patients.patient_id`
- `billing.visit_id` references `visits.visit_id`

Relationship model:

```text
patients 1-to-many visits 1-to-1 billing
```


In [ ]:
print((ROOT / 'sql' / '01_schema.sql').read_text())


## Reusable Business Views

These views simplify downstream operational and billing reporting.


In [ ]:
print((ROOT / 'sql' / '02_views.sql').read_text())


# Phase 1 Index Strategy

The primary keys automatically index `patients.patient_id`, `visits.visit_id`, and `billing.bill_id`.

Indexes are added on columns that are frequently used in:

- `JOIN` conditions, such as `visits.patient_id` and `billing.visit_id`
- `WHERE` filters, such as `risk_score`, `claim_status`, and `billed_amount`
- `GROUP BY` analysis, such as `department`, `city`, `insurance_provider`, and `doctor_id`
- future dashboard filters, such as `visit_date`

## How Indexes Work

Without an index, the database may scan every row in a table to find matching records. That is acceptable for a small CSV, but it becomes slow as the hospital data grows.

An index is like a sorted lookup structure. When a query filters or joins on an indexed column, SQLite can jump directly to matching values instead of reading the full table.

Example:

```sql
SELECT *
FROM visits
WHERE patient_id = 756;
```

Because `visits.patient_id` has an index, SQLite can quickly find visits for patient `756`.

Another example:

```sql
SELECT *
FROM visits AS v
JOIN billing AS b
    ON b.visit_id = v.visit_id;
```

Because `billing.visit_id` is indexed, the database can quickly find the billing row for each visit.

Indexes improve read performance, but they slightly slow down inserts/updates because the index also has to be maintained. For this analytics layer, that tradeoff is appropriate because the main workload is reporting and analysis.

| Index | Helps Queries | Reason |
| --- | --- | --- |
| `idx_patients_city` | Average visits per patient by city | Speeds patient grouping and city-level aggregation. |
| `idx_patients_insurance_provider` | Insurance billed amount, rejection rate, payment delay | Speeds grouping after joining visits/billing to patient insurance. |
| `idx_visits_patient_id` | Patient visit counts, visit-to-patient joins, FK enforcement | Speeds joins from visits to patients and validates visit imports. |
| `idx_visits_department` | Department volume, length of stay, realization ratio | Speeds department grouping and sorting. |
| `idx_visits_department_risk` | High Risk percentage per department | Covers department/risk grouping and conditional risk counts. |
| `idx_visits_doctor_risk` | Doctors with the most High Risk visits | Speeds filtering by `risk_score = 'High'` and grouping by doctor. |
| `idx_visits_visit_date` | Future date-window dashboards | Supports common leadership reporting by period. |
| `idx_billing_visit_id` | Visit billing joins, missing billing checks | Speeds joins from visits to billing and supports the one-bill-per-visit relationship. |
| `idx_billing_claim_status` | Rejection-rate analysis | Speeds claim-status filtering and conditional aggregation. |
| `idx_billing_billed_amount` | High billed / zero approved leakage checks | Speeds top-value billing analysis and percentile/ranking workflows. |

## Query-to-Index Mapping

| Required Query | Helpful Indexes | Why These Indexes Help |
| --- | --- | --- |
| Top 10 departments by total visit volume | `idx_visits_department` | Query groups visits by `department`. |
| Top 5 departments with highest average length of stay | `idx_visits_department` | Query groups visits by `department` before calculating average `length_of_stay_hours`. |
| Percentage of High Risk visits per department | `idx_visits_department_risk` | Query groups by `department` and checks `risk_score = 'High'`. Composite index keeps both columns together. |
| Average number of visits per patient by city | `idx_patients_city`, `idx_visits_patient_id` | Query groups patients by `city` and joins visits through `patient_id`. |
| Doctors handling highest number of High Risk visits | `idx_visits_doctor_risk` | Query filters/counts High Risk visits and groups by `doctor_id`. |
| Top 10 insurance providers by total billed amount | `idx_patients_insurance_provider`, `idx_visits_patient_id`, `idx_billing_visit_id` | Query joins patients, visits, and billing, then groups by insurer. |
| Insurance providers with highest claim rejection rate | `idx_patients_insurance_provider`, `idx_billing_claim_status`, `idx_billing_visit_id` | Query groups by insurer and counts `claim_status = 'Rejected'`. |
| Average payment delay by insurance provider | `idx_patients_insurance_provider`, `idx_billing_visit_id` | Query joins billing to patient insurance and groups by insurer. |
| Revenue realization ratio by department | `idx_visits_department`, `idx_billing_visit_id` | Query groups by department and joins billing amounts by visit. |
| High billed amount but zero or missing approved amount | `idx_billing_billed_amount`, `idx_billing_visit_id` | Query ranks or filters by `billed_amount` and joins back to visit details. |
| Visits without corresponding billing record | `idx_billing_visit_id` | Query left joins visits to billing using `visit_id`. |
| Billing records without corresponding visit | Primary key index on `visits.visit_id` | Query checks whether each billing `visit_id` exists in visits. |
| Duplicate patient IDs | Primary key index on `patients.patient_id`; raw table scan for source duplicates | Final table prevents duplicates; raw table check scans source data to detect duplicate IDs before enforcement. |
| Missing or invalid length of stay | No dedicated index | This is a full data-quality scan of `raw_visits`; an index is not useful for checking all rows. |
| Missing or invalid payment days | No dedicated index | This is a full data-quality scan of `raw_billing`; an index is not useful for checking all rows. |
| Visits linked to patients with missing insurance provider | `idx_visits_patient_id`, `idx_patients_insurance_provider` | Query joins visits to patients and filters missing/blank insurer values. |

For this data volume, full scans are still acceptable. These indexes document the access paths that matter once the hospital dataset grows or the same views power dashboards.


# SQL Analysis Queries

The full query file is shown below. These queries produce the operational and financial analysis results in the following section.


In [ ]:
print((ROOT / 'sql' / '03_analysis_queries.sql').read_text())


# Data Quality Queries

The full data-quality and integrity-check query file is shown below.


In [ ]:
print((ROOT / 'sql' / '04_data_quality_checks.sql').read_text())


# Phase 1 Analysis Results

Generated from `hospital_analytics.db` after running:

```bash
python3 scripts/build_sqlite_db.py
```

The SQL used for these results is stored in `sql/03_analysis_queries.sql` and `sql/04_data_quality_checks.sql`.

## Operational Analysis

### 1. Top 10 Departments by Total Visit Volume

The dataset contains 6 departments, so all available departments are shown.

| rank | department | total_visits |
| ---: | --- | ---: |
| 1 | General | 4,228 |
| 2 | ER | 4,220 |
| 3 | Neurology | 4,165 |
| 4 | Orthopedics | 4,164 |
| 5 | Cardiology | 4,159 |
| 6 | ICU | 4,064 |

Observation: Visit volume is evenly distributed across departments. General has the highest volume, while ICU has the lowest.

### 2. Top 5 Departments with Highest Average Length of Stay

| rank | department | avg_length_of_stay_hours |
| ---: | --- | ---: |
| 1 | Neurology | 19.72 |
| 2 | Orthopedics | 19.66 |
| 3 | Cardiology | 19.60 |
| 4 | ER | 19.53 |
| 5 | General | 19.43 |

Observation: Neurology has the highest average length of stay, which may indicate higher case complexity or slower patient throughput.

### 3. Percentage of High Risk Visits per Department

| department | total_visits | high_risk_visits | high_risk_visit_pct |
| --- | ---: | ---: | ---: |
| ICU | 4,064 | 845 | 20.79 |
| ER | 4,220 | 872 | 20.66 |
| Neurology | 4,165 | 846 | 20.31 |
| Orthopedics | 4,164 | 842 | 20.22 |
| General | 4,228 | 839 | 19.84 |
| Cardiology | 4,159 | 790 | 18.99 |

Observation: ICU has the highest High Risk visit percentage at 20.79%, followed closely by ER.

### 4. Average Number of Visits per Patient by City

| city | avg_visits_per_patient |
| --- | ---: |
| Pune | 5.08 |
| Hyderabad | 5.03 |
| Bangalore | 5.01 |
| Mumbai | 5.01 |
| Chennai | 4.96 |
| Delhi | 4.91 |

Observation: Pune patients average the most visits per patient. Delhi has the lowest average in this dataset.

### 5. Doctors Handling the Highest Number of High Risk Visits

| rank | doctor_id | high_risk_visits |
| ---: | ---: | ---: |
| 1 | 174 | 71 |
| 2 | 198 | 69 |
| 3 | 169 | 68 |
| 4 | 177 | 67 |
| 5 | 105 | 65 |
| 6 | 135 | 65 |
| 7 | 180 | 64 |
| 8 | 188 | 64 |
| 9 | 131 | 62 |
| 10 | 108 | 61 |

Observation: Doctor 174 handled the highest number of High Risk visits.

## Financial Analysis

### 1. Top 10 Insurance Providers by Total Billed Amount

The dataset contains 4 insurance providers, so all available providers are shown.

| rank | insurance_provider | total_billed_amount |
| ---: | --- | ---: |
| 1 | MediCareX | 134,591,163.08 |
| 2 | CareOne | 130,707,992.64 |
| 3 | HealthPlus | 130,180,740.75 |
| 4 | SecureLife | 126,289,039.58 |

Observation: MediCareX has the highest total billed amount.

### 2. Top 5 Insurance Providers with Highest Claim Rejection Rate

The dataset contains 4 insurance providers, so all available providers are shown.

| rank | insurance_provider | total_claims | rejected_claims | claim_rejection_rate_pct |
| ---: | --- | ---: | ---: | ---: |
| 1 | SecureLife | 5,965 | 936 | 15.69 |
| 2 | MediCareX | 6,532 | 996 | 15.25 |
| 3 | HealthPlus | 6,220 | 931 | 14.97 |
| 4 | CareOne | 6,283 | 934 | 14.87 |

Observation: SecureLife has the highest rejection rate at 15.69%.

### 3. Average Payment Delay by Insurance Provider

| insurance_provider | avg_payment_days |
| --- | ---: |
| HealthPlus | 13.08 |
| SecureLife | 13.08 |
| CareOne | 13.03 |
| MediCareX | 13.01 |

Observation: Average payment delay is very similar across providers, ranging from 13.01 to 13.08 days.

### 4. Revenue Realization Ratio by Department

Revenue realization ratio is calculated as:

```text
approved_amount / billed_amount
```

| department | total_approved_amount | total_billed_amount | revenue_realization_ratio |
| --- | ---: | ---: | ---: |
| Cardiology | 63,705,806.68 | 86,071,256.19 | 0.7402 |
| ER | 65,672,329.38 | 88,686,960.35 | 0.7405 |
| Neurology | 64,708,778.69 | 87,310,048.09 | 0.7411 |
| General | 64,690,870.95 | 87,131,451.86 | 0.7425 |
| Orthopedics | 65,211,585.83 | 87,811,455.80 | 0.7426 |
| ICU | 63,166,516.84 | 84,757,763.76 | 0.7453 |

Observation: Cardiology has the lowest revenue realization ratio, while ICU has the highest. All departments realize roughly 74% of billed amounts.

### 5. Visits Where Billed Amount Is High but Approved Amount Is Zero or Missing

For this analysis, "high billed amount" is defined as visits in the top 5% by billed amount. The query found 72 flagged visits. The flagged billed amount range is 44,175.34 to 78,054.79.

Top flagged visits by billed amount:

| visit_id | patient_id | department | insurance_provider | billed_amount | approved_amount | claim_status |
| ---: | ---: | --- | --- | ---: | ---: | --- |
| 1570 | 2685 | General | HealthPlus | 78,054.79 | missing | Paid |
| 18381 | 3817 | General | HealthPlus | 68,213.53 | 0.00 | Rejected |
| 15092 | 4096 | Neurology | CareOne | 66,410.33 | missing | Paid |
| 2638 | 4581 | Neurology | CareOne | 65,167.44 | missing | Paid |
| 8089 | 1774 | ER | HealthPlus | 62,661.81 | missing | Paid |
| 19310 | 931 | ICU | MediCareX | 60,510.22 | missing | Paid |
| 24623 | 2828 | Neurology | SecureLife | 58,275.40 | missing | Paid |
| 18189 | 3339 | Neurology | HealthPlus | 57,521.12 | missing | Paid |
| 14702 | 4844 | Orthopedics | HealthPlus | 56,848.91 | missing | Paid |
| 5131 | 2167 | Orthopedics | MediCareX | 56,736.52 | missing | Paid |

Observation: These visits are revenue leakage risks because billed amounts are large, but approval is zero or unavailable.

## Data Quality and Integrity Checks

### 1. Visits Without a Corresponding Billing Record

| check_name | issue_count |
| --- | ---: |
| visits_without_billing | 0 |

Result: Every visit in the final `visits` table has a corresponding billing record.

### 2. Billing Records Without a Corresponding Visit

| check_name | issue_count |
| --- | ---: |
| billing_without_visit | 0 |

Result: Every source billing record references a valid visit.

### 3. Patients with Duplicate `patient_id` Values

| check_name | issue_count |
| --- | ---: |
| duplicate_patient_ids | 0 |

Result: No duplicate patient IDs were found in the source patient data.

### 4. Records with Missing or Invalid `length_of_stay_hours` or `payment_days`

| data_quality_check | issue_count |
| --- | ---: |
| missing_or_invalid_length_of_stay_hours | 0 |
| missing_or_invalid_payment_days | 790 |

Result: No invalid length-of-stay records were found. However, 790 billing records have missing or invalid `payment_days`.

Sample records with missing or invalid `payment_days`:

| bill_id | visit_id | claim_status | billed_amount | approved_amount | payment_days |
| ---: | ---: | --- | ---: | ---: | --- |
| 3 | 3 | Paid | 5,038.97 | 5,038.97 | missing |
| 22 | 22 | Paid | 31,418.34 | 31,418.34 | missing |
| 41 | 41 | Paid | 12,924.18 | 12,924.18 | missing |
| 66 | 66 | Paid | 28,126.25 | 28,126.25 | missing |
| 109 | 109 | Pending | 1,233.62 | 911.46 | missing |
| 120 | 120 | Paid | 23,218.37 | missing | missing |
| 133 | 133 | Paid | 500.00 | 500.00 | missing |
| 173 | 173 | Pending | 11,493.55 | 8,735.77 | missing |
| 181 | 181 | Paid | 49,363.32 | 49,363.32 | missing |
| 237 | 237 | Paid | 66,037.73 | 66,037.73 | missing |

### 5. Visits Linked to Patients with Missing Insurance Provider Information

| check_name | issue_count |
| --- | ---: |
| visits_missing_patient_insurance | 0 |

Result: No visits are linked to patients with missing insurance provider information.

## Summary

The relational layer loaded successfully with 5,000 patients, 25,000 visits, and 25,000 billing records. Referential integrity is clean: no orphan visits, no orphan billing rows, and no duplicate patient IDs were detected. The main data-quality issue is missing `payment_days` in 790 billing records. The main financial risk area is the set of 72 high-billed visits where `approved_amount` is zero or missing.


## Execute Query Files as a Validation Check

This confirms that the analysis and data-quality SQL scripts run successfully against the generated database.


In [ ]:
conn = sqlite3.connect(DB_PATH)
try:
    for sql_file in ['sql/03_analysis_queries.sql', 'sql/04_data_quality_checks.sql']:
        conn.executescript((ROOT / sql_file).read_text())
        print(f'{sql_file}: ok')
finally:
    conn.close()
